# Orbit Wars — Exploration Notebook

Lightweight notebook for game exploration, quick experiments, and visualisation.
Heavy lifting (training, evaluation) lives in `training/` and `scripts/`.

| Section | What it does |
|---------|-------------|
| 1. Setup | Install deps + add repo root to path |
| 2. Game demo | Watch baseline vs random |
| 3. Quick train | Short PPO run importable from training/ |
| 4. Evaluation | Win-rate table |
| 5. Submission | Generate submission.py |

## 1. Setup

In [ ]:
import os, sys

def detect_env():
    if os.path.exists('/kaggle/input'):
        return 'kaggle'
    try:
        import google.colab  # noqa: F401
        return 'colab'
    except ImportError:
        return 'local'

ENV = detect_env()
print(f'Runtime: {ENV}')

In [ ]:
if ENV in ('kaggle', 'colab'):
    os.system('pip install -q --upgrade "kaggle-environments>=1.28.0"')
    os.system('pip install -q "stable-baselines3[extra]>=2.3" gymnasium pyyaml')

if ENV == 'colab':
    # Clone repo so local modules are importable.
    # Replace with your actual repo URL.
    REPO_URL = 'https://github.com/YOUR_USERNAME/orbit_wars.git'
    if not os.path.exists('orbit_wars'):
        os.system(f'git clone {REPO_URL} orbit_wars')
    os.chdir('orbit_wars')

elif ENV == 'kaggle':
    # If you uploaded the repo as a dataset, add its path here.
    DATASET_PATH = '/kaggle/input/orbit-wars-code'
    if os.path.exists(DATASET_PATH):
        sys.path.insert(0, DATASET_PATH)

# Ensure repo root is always on path (works for local & after chdir above)
repo_root = os.path.abspath('.')
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

print('Working dir:', os.getcwd())

In [ ]:
import math, time
import numpy as np
import torch
from kaggle_environments import make

print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU: {torch.cuda.get_device_name(0)}')

## 2. Game Demo

In [ ]:
from agents.baseline import agent as baseline_agent

env = make('orbit_wars', debug=True)
env.run([baseline_agent, 'random'])

final = env.steps[-1]
print(f'Steps: {len(env.steps)}')
for i, s in enumerate(final):
    print(f'  Player {i}: reward={s.reward}  status={s.status}')

In [ ]:
env.render(mode='ipython', width=800, height=600)

In [ ]:
from evaluation.evaluate import run_games, print_results

r = run_games(baseline_agent, 'random', n_games=10)
print_results('baseline', 'random', r)

## 3. Gymnasium Environment Check

In [ ]:
from envs.orbit_wars_env import OrbitWarsEnv, OBS_DIM

env_rl = OrbitWarsEnv(opponent='baseline', reward_shaping=True)
obs, _ = env_rl.reset(seed=42)
print(f'obs shape : {obs.shape}  dtype={obs.dtype}')
print(f'obs range : [{obs.min():.3f}, {obs.max():.3f}]')
print(f'action    : {env_rl.action_space}')

total_r = 0.0
for _ in range(20):
    a = env_rl.action_space.sample()
    obs, r, done, _, _ = env_rl.step(a)
    total_r += r
    if done:
        break
print(f'20-step reward: {total_r:.4f}')

## 4. Quick Training Run

This cell runs a minimal training loop directly from the `training` module.
For serious runs use the CLI:
```
python scripts/train.py --config configs/ppo_default.yaml
python scripts/train.py --config configs/ppo_default.yaml --set training.total_timesteps=1000000
```

In [ ]:
from training.train import load_config, apply_dotted_overrides, train

config = load_config('configs/ppo_default.yaml')

# Reduce timesteps for a quick smoke test
config = apply_dotted_overrides(config, [
    'training.total_timesteps=20000',
    'env.n_envs=2',
    'eval.freq=5000',
    'eval.n_episodes=5',
    'output.run_name=notebook_quick',
])

model, checkpoint_dir = train(config)
print(f'\nCheckpoints saved to: {checkpoint_dir}')

## 5. Evaluation

In [ ]:
from pathlib import Path
from agents.rl_agent import RLAgent
from evaluation.evaluate import benchmark

best = checkpoint_dir / 'best_model.zip'
if best.exists():
    rl_agent = RLAgent(best)
    benchmark(rl_agent, agent_name='rl_agent', n_games=10)
else:
    print('No best_model.zip yet — run more timesteps or check checkpoint_dir.')

## 6. Generate Submission

Or from the CLI:
```
python scripts/submit.py --baseline
python scripts/submit.py --model outputs/checkpoints/<run>/best_model.zip --verify
```

In [ ]:
from agents.rl_agent import export_submission

# Option A — baseline
export_submission(None, 'outputs/submissions/submission_baseline.py', mode='baseline')

In [ ]:
# Option B — trained RL agent (run after Section 4)
if best.exists():
    export_submission(best, 'outputs/submissions/submission_rl.py', mode='rl')
else:
    print('Train a model first (Section 4).')